In [1]:
# Cell 1
!pip install -U bitsandbytes transformers accelerate zeus-ml scikit-learn tqdm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 87.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 94.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 13.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym

In [2]:
# Cell 1b
from kaggle_secrets import UserSecretsClient
import os
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [3]:
# Cell 2
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [4]:
# Cell 3
import torch
print("CUDA:", torch.cuda.is_available(), "| Device:", torch.cuda.get_device_name(0))

CUDA: True | Device: Tesla T4


In [5]:
# Cell 4 — GLM-4.1V, eager attention, NF4
from transformers import AutoProcessor, Glm4vForConditionalGeneration, BitsAndBytesConfig

MODEL_ID = "THUDM/GLM-4.1V-9B-Thinking"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_use_double_quant=True)

processor = AutoProcessor.from_pretrained(MODEL_ID, use_fast=True)
model = Glm4vForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map={"": 0},
    attn_implementation="eager",
)
model.eval()
print("Loaded. Layers:", model.config.text_config.num_hidden_layers)

preprocessor_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/922 [00:00<?, ?B/s]

[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

... (2 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/704 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Loaded. Layers: 40


In [6]:
# Cell 5 — data paths
import os, json
from PIL import Image
from tqdm.auto import tqdm

DATA_DIR = "/kaggle/input/notebooks/minhajurrahman223050/ecovlm-data-prep/vqav2_subset"
IMG_DIR = os.path.join(DATA_DIR, "images")
with open(os.path.join(DATA_DIR, "metadata.json")) as f:
    SAMPLES = json.load(f)
print("Samples:", len(SAMPLES))

Samples: 1000


In [7]:
# Cell 6 — find the text decoder and inspect its forward source
import torch, types
import numpy as np
from transformers.modeling_outputs import BaseModelOutputWithPast
from transformers.cache_utils import DynamicCache
from transformers.masking_utils import create_causal_mask

text_model = model.model.language_model
IMG_TOKEN_ID_GLM = model.config.image_token_id

CHECKPOINT_LAYERS_GLM = [11, 16, 20]
EARLY_WINDOW_GLM = list(range(3, 12))
ALL_SCORE_LAYERS_GLM = list(range(3, 21))
DROP_FRAC_GLM = 0.5
MAX_STEPS_GLM = 350
KEEP_PCTILE = 70
DROP_PCTILE = 40

def extract_final_answer_glm(raw_text: str) -> str:
    import re
    if "</think>" in raw_text:
        raw_text = raw_text.split("</think>", 1)[1]
    m = re.search(r"<\|begin_of_box\|>(.*?)<\|end_of_box\|>", raw_text, re.DOTALL)
    if m:
        return m.group(1).strip()
    return raw_text.strip()

def apply_checkpoint_stub_glm(hidden_states, position_embeddings, causal_mask, survivor_idx):
    hidden_states = hidden_states[:, survivor_idx, :]
    cos, sin = position_embeddings
    cos = cos[..., survivor_idx, :]
    sin = sin[..., survivor_idx, :]
    position_embeddings = (cos, sin)
    causal_mask = causal_mask[:, :, survivor_idx, :][:, :, :, survivor_idx]
    return hidden_states, position_embeddings, causal_mask

attn_capture_glm = {}

def make_capture_hook_glm(layer_idx):
    def hook(module, inputs, output):
        if isinstance(output, tuple) and len(output) > 1 and output[1] is not None:
            attn_capture_glm[layer_idx] = output[1][0].float().mean(dim=0).detach().cpu().numpy()
        return output
    return hook

def register_all_hooks_glm(text_model, layers):
    return [text_model.layers[i].self_attn.register_forward_hook(make_capture_hook_glm(i)) for i in layers]

def score_and_gate(candidate_local_idx, mean_vals, var_vals, force_resolve=False):
    n = len(candidate_local_idx)
    if n == 0:
        return np.array([], dtype=int), np.array([], dtype=int), np.array([], dtype=int)
    mean_cut = np.percentile(mean_vals, KEEP_PCTILE)
    high_mean_mask = mean_vals >= mean_cut
    keep_idx = candidate_local_idx[high_mean_mask]
    low_mean_local = candidate_local_idx[~high_mean_mask]
    low_mean_var = var_vals[~high_mean_mask]
    if len(low_mean_local) == 0:
        return keep_idx, np.array([], dtype=int), np.array([], dtype=int)
    if force_resolve:
        var_cut = np.median(low_mean_var)
        drop_mask = low_mean_var <= var_cut
        drop_idx = low_mean_local[drop_mask]
        keep_idx = np.concatenate([keep_idx, low_mean_local[~drop_mask]])
        defer_idx = np.array([], dtype=int)
    else:
        var_cut = np.percentile(low_mean_var, DROP_PCTILE)
        drop_mask = low_mean_var <= var_cut
        drop_idx = low_mean_local[drop_mask]
        defer_idx = low_mean_local[~drop_mask]
    return keep_idx, drop_idx, defer_idx

print("Setup complete:", type(text_model), IMG_TOKEN_ID_GLM)

Setup complete: <class 'transformers.models.glm4v.modeling_glm4v.Glm4vTextModel'> 151343


In [8]:
# Cell 7 — dap_forward_glm_v3 (full adaptive 3-checkpoint DAP for GLM)

def dap_forward_glm_v3(self, input_ids=None, attention_mask=None, position_ids=None,
                        past_key_values=None, inputs_embeds=None, use_cache=None, **kwargs):
    if use_cache and past_key_values is None and not torch.jit.is_tracing():
        past_key_values = DynamicCache(config=self.config)
    if inputs_embeds is None:
        inputs_embeds = self.embed_tokens(input_ids)

    if position_ids is None:
        past_seen_tokens = past_key_values.get_seq_length() if past_key_values is not None else 0
        position_ids = torch.arange(inputs_embeds.shape[1], device=inputs_embeds.device) + past_seen_tokens
        position_ids = position_ids.view(1, 1, -1).expand(3, inputs_embeds.shape[0], -1)
    elif position_ids.ndim == 2:
        position_ids = position_ids[None, ...].expand(3, position_ids.shape[0], -1)

    text_position_ids = None
    if position_ids.ndim == 3 and position_ids.shape[0] == 4:
        text_position_ids = position_ids[0]
        position_ids = position_ids[1:]

    mask_kwargs = {"config": self.config, "inputs_embeds": inputs_embeds,
                   "attention_mask": attention_mask, "past_key_values": past_key_values,
                   "position_ids": text_position_ids}
    causal_mask = create_causal_mask(**mask_kwargs)

    hidden_states = inputs_embeds
    position_embeddings = self.rotary_emb(hidden_states, position_ids=position_ids)
    seq_len = hidden_states.shape[1]
    survivor_idx = torch.arange(seq_len, device=hidden_states.device)

    vis_start, vis_end, text_start = self._dap_vis_range
    deferred_local = np.arange(vis_start, vis_end)

    for i, decoder_layer in enumerate(self.layers):
        hidden_states = decoder_layer(
            hidden_states, attention_mask=causal_mask, position_ids=text_position_ids,
            past_key_values=past_key_values, position_embeddings=position_embeddings, **kwargs,
        )

        if i in CHECKPOINT_LAYERS_GLM:
            ckpt_idx = CHECKPOINT_LAYERS_GLM.index(i)
            is_final = (ckpt_idx == len(CHECKPOINT_LAYERS_GLM) - 1)

            window_layers = [l for l in attn_capture_glm.keys()]
            stacked = np.stack([attn_capture_glm[l][text_start:, deferred_local].mean(axis=0) for l in window_layers])
            mean_vals = stacked.mean(axis=0)
            var_vals = stacked.var(axis=0)
            attn_capture_glm.clear()

            keep_local, drop_local, defer_local = score_and_gate(
                deferred_local, mean_vals, var_vals, force_resolve=is_final
            )

            seq_len_now = hidden_states.shape[1]
            drop_set = set(drop_local.tolist())
            keep_mask = torch.ones(seq_len_now, dtype=torch.bool, device=hidden_states.device)
            for d in drop_set:
                keep_mask[d] = False
            survivor_idx = torch.nonzero(keep_mask).squeeze(-1)

            print(f"  [ckpt @ layer {i}] keep={len(keep_local)} drop={len(drop_local)} defer={len(defer_local)} (pool was {len(deferred_local)})")

            hidden_states, position_embeddings, causal_mask = apply_checkpoint_stub_glm(
                hidden_states, position_embeddings, causal_mask, survivor_idx
            )

            surv_list = survivor_idx.cpu().numpy()
            remap = {old: new for new, old in enumerate(surv_list)}
            deferred_local = np.array([remap[d] for d in defer_local if d in remap], dtype=int)
            text_start = remap.get(text_start, text_start - len(drop_set))

    hidden_states = self.norm(hidden_states)
    return BaseModelOutputWithPast(last_hidden_state=hidden_states, past_key_values=past_key_values)

print("dap_forward_glm_v3 defined.")

dap_forward_glm_v3 defined.


In [9]:
# cell 8
from zeus.monitor import ZeusMonitor
import time

monitor = ZeusMonitor(gpu_indices=[0])
print("Zeus initialized.")

print("Measuring idle baseline (30 seconds)...")
idle_readings = []
for _ in range(300):
    idle_readings.append(monitor.gpus.getInstantPowerUsage(0) / 1000)
    time.sleep(0.1)

IDLE_POWER_W = sum(idle_readings) / len(idle_readings)
print(f"Idle baseline: {IDLE_POWER_W:.3f} W")

/opt/rocm/lib/libamd_smi.so: cannot open shared object file: No such file or directory
Unable to find libamd_smi.so library try installing amd-smi-lib from your package manager
[2026-07-28 18:10:52,081] [zeus.device.gpu.nvidia](nvidia.py:47) pynvml is available and initialized.
[2026-07-28 18:10:52,082] [zeus.device.cpu.rapl](rapl.py:137) RAPL is not supported on this CPU.
[2026-07-28 18:10:52,083] [zeus.monitor.energy](energy.py:209) Monitoring GPU indices [0].
[2026-07-28 18:10:52,083] [zeus.monitor.energy](energy.py:210) Monitoring CPU indices []
Zeus initialized.
Measuring idle baseline (30 seconds)...
Idle baseline: 27.643 W


In [10]:
# cell 9

import re

def vqa_accuracy(predicted, ground_truth_answers):
    def normalise(s):
        s = s.lower().strip()
        s = re.sub(r"[^\w\s]", "", s)
        for a in ["a ", "an ", "the "]:
            if s.startswith(a): s = s[len(a):]
        return s.strip()
    p = normalise(predicted)
    m = sum(1 for a in ground_truth_answers if normalise(a) == p)
    return min(m / 3, 1.0)


def run_dap_test_glm_energy(sample, use_dap):
    image = Image.open(os.path.join(IMG_DIR, sample["image_file"])).convert("RGB")
    messages = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": f"{sample['question']} Answer the question using a single word or phrase."}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt").to(model.device)

    ids = inputs["input_ids"][0]
    vis_mask = (ids == IMG_TOKEN_ID_GLM)
    vis_idx = torch.nonzero(vis_mask).squeeze(-1)
    vis_start, vis_end = vis_idx[0].item(), vis_idx[-1].item() + 1
    text_start = vis_end

    with torch.no_grad():
        inputs_embeds = model.get_input_embeddings()(inputs["input_ids"])
        image_embeds = model.model.get_image_features(
            inputs["pixel_values"], inputs["image_grid_thw"], return_dict=True
        ).pooler_output
        image_embeds = torch.cat(image_embeds, dim=0).to(inputs_embeds.device, inputs_embeds.dtype)
        image_mask, _ = model.model.get_placeholder_mask(inputs["input_ids"], inputs_embeds, image_features=image_embeds)
        inputs_embeds = inputs_embeds.masked_scatter(image_mask, image_embeds)

        position_ids = model.model.compute_3d_position_ids(
            input_ids=inputs["input_ids"], image_grid_thw=inputs["image_grid_thw"],
            inputs_embeds=inputs_embeds, mm_token_type_ids=inputs["mm_token_type_ids"],
        )

        hooks = []
        if use_dap:
            attn_capture_glm.clear()
            hooks = register_all_hooks_glm(text_model, ALL_SCORE_LAYERS_GLM)
            text_model._dap_vis_range = (vis_start, vis_end, text_start)
            original_forward = text_model.forward
            text_model.forward = types.MethodType(dap_forward_glm_v3, text_model)

        window_name = f"{'dap' if use_dap else 'base'}_{sample['sample_idx']}"
        monitor.begin_window(window_name)
        t0 = time.perf_counter()

        current_embeds = inputs_embeds
        current_pos_ids = position_ids
        generated = None
        for step in range(MAX_STEPS_GLM):
            out = text_model(inputs_embeds=current_embeds, position_ids=current_pos_ids, use_cache=False)
            next_logits = model.lm_head(out.last_hidden_state[:, -1, :])
            next_tok = next_logits.argmax(-1)
            if next_tok.item() == processor.tokenizer.eos_token_id:
                break
            next_embed = model.get_input_embeddings()(next_tok.unsqueeze(0))
            current_embeds = torch.cat([current_embeds, next_embed], dim=1)
            next_pos = current_pos_ids[..., -1:] + 1
            current_pos_ids = torch.cat([current_pos_ids, next_pos], dim=-1)
            generated = next_tok if generated is None else torch.cat([generated, next_tok])

        elapsed = time.perf_counter() - t0
        measurement = monitor.end_window(window_name)

        if use_dap:
            text_model.forward = original_forward
            for h in hooks: h.remove()

    tokens_generated = 0 if generated is None else generated.shape[0]
    raw = "" if generated is None else processor.tokenizer.decode(generated, skip_special_tokens=True).strip()
    predicted = extract_final_answer_glm(raw)

    energy_gross = measurement.total_energy
    avg_power = energy_gross / elapsed if elapsed > 0 else 0
    energy_net = max((avg_power - IDLE_POWER_W) * elapsed, 0)

    return {"predicted": predicted, "tokens": tokens_generated, "elapsed": elapsed,
            "energy_net": energy_net, "energy_gross": energy_gross}


N_ENERGY_GLM = 3
results_glm = []

for idx, s in enumerate(SAMPLES[:N_ENERGY_GLM]):
    base = run_dap_test_glm_energy(s, use_dap=False)
    dap = run_dap_test_glm_energy(s, use_dap=True)
    results_glm.append({
        "question": s["question"],
        "base_acc": vqa_accuracy(base["predicted"], s["answers"]),
        "dap_acc": vqa_accuracy(dap["predicted"], s["answers"]),
        "base_energy": base["energy_net"], "dap_energy": dap["energy_net"],
        "base_elapsed": base["elapsed"], "dap_elapsed": dap["elapsed"],
    })
    print(f"  sample {idx+1}/{N_ENERGY_GLM} done")

import statistics as st
base_e = st.mean(r["base_energy"] for r in results_glm)
dap_e = st.mean(r["dap_energy"] for r in results_glm)
base_acc = st.mean(r["base_acc"] for r in results_glm)
dap_acc = st.mean(r["dap_acc"] for r in results_glm)

print("\n" + "="*60)
print("GLM ENERGY RESULT")
print("="*60)
print(f"N = {N_ENERGY_GLM}")
print(f"Baseline energy_net (J): {base_e:.2f}")
print(f"DAP energy_net (J):      {dap_e:.2f}")
print(f"Energy reduction:        {100*(1 - dap_e/base_e):.1f}%")
print(f"Baseline accuracy: {base_acc:.4f}  |  DAP accuracy: {dap_acc:.4f}")

[2026-07-28 18:11:37,164] [zeus.utils.framework](framework.py:25) PyTorch with CUDA support is available.
  [ckpt @ layer 11] keep=70 drop=66 defer=98 (pool was 234)
  [ckpt @ layer 16] keep=30 drop=27 defer=41 (pool was 98)
  [ckpt @ layer 20] keep=27 drop=14 defer=0 (pool was 41)
  [ckpt @ layer 11] keep=70 drop=66 defer=98 (pool was 234)
  [ckpt @ layer 16] keep=30 drop=27 defer=41 (pool was 98)
  [ckpt @ layer 20] keep=27 drop=14 defer=0 (pool was 41)
  [ckpt @ layer 11] keep=70 drop=66 defer=98 (pool was 234)
  [ckpt @ layer 16] keep=30 drop=27 defer=41 (pool was 98)
  [ckpt @ layer 20] keep=27 drop=14 defer=0 (pool was 41)
  [ckpt @ layer 11] keep=70 drop=66 defer=98 (pool was 234)
  [ckpt @ layer 16] keep=30 drop=27 defer=41 (pool was 98)
  [ckpt @ layer 20] keep=27 drop=14 defer=0 (pool was 41)
  [ckpt @ layer 11] keep=70 drop=66 defer=98 (pool was 234)
  [ckpt @ layer 16] keep=30 drop=27 defer=41 (pool was 98)
  [ckpt @ layer 20] keep=27 drop=14 defer=0 (pool was 41)
  [ckpt @